In [1]:
import sys
import sklearn
import skopt
import pandas as pd
import numpy as np
import joblib
 
from sklearn.neural_network import MLPRegressor
 
from sklearn.model_selection import GroupKFold
 
from sklearn.preprocessing import StandardScaler
 
from sklearn.pipeline import Pipeline
 
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
 
from skopt import BayesSearchCV
from skopt import Optimizer
from skopt.space import (
    Real,
    Integer,
    Categorical
)

In [2]:
print(sys.version)
print(sklearn.__version__)
print(skopt.__version__)

3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:29:05) [MSC v.1942 64 bit (AMD64)]
1.9.0
0.10.2


In [3]:
# =====================================================
# LOAD DATA
# =====================================================
 
df = pd.read_csv("Final_for_ML1.csv")
 
df["Experiment_ID"] = (
    df["Source#"].astype(str)
    + "_"
    + df["Database#"].astype(str)
)
 
df = pd.get_dummies(
    df,
    columns=["Glycerol_Source"],
    drop_first=False
)
 
groups = df["Experiment_ID"]
df.head()

,Source#,Database#,t,X,S,dXdt,dSdt,C_N_Ratio,Working_Volume_L,Seed_Volume_L,...,Temp_C,pH,KH2PO4,Na2HPO4,MgSO4,NaCl,CaCl2,Experiment_ID,Glycerol_Source_pure,Glycerol_Source_raw
0,5,16,0.0,11.470588,20.374332,0.571556,-1.215865,3.793506,1.5,150,...,30,3.5,0.125,0.0,0.2442,0.0,0.0,5_16,False,True
1,5,16,0.2,11.584899,20.131159,0.575805,-1.224773,3.748230,1.5,150,...,30,3.5,0.125,0.0,0.2442,0.0,0.0,5_16,False,True
2,5,16,0.4,11.700910,19.884422,0.584410,-1.242810,3.702290,1.5,150,...,30,3.5,0.125,0.0,0.2442,0.0,0.0,5_16,False,True
3,5,16,0.6,11.818663,19.634035,0.593233,-1.261298,3.655670,1.5,150,...,30,3.5,0.125,0.0,0.2442,0.0,0.0,5_16,False,True
4,5,16,0.8,11.938203,19.379903,0.602283,-1.280257,3.608353,1.5,150,...,30,3.5,0.125,0.0,0.2442,0.0,0.0,5_16,False,True


In [4]:
# =====================================================
# TARGET
# =====================================================

target = "dXdt"
 
exclude = [
    "Source#",
    "Database#",
    "dXdt",
    "Experiment_ID"
]
 
features = [
    c for c in df.columns
    if c not in exclude
]
 
X = df[features]
y = df[target]

In [5]:
# =====================================================
# PIPELINE
# =====================================================
 
pipeline = Pipeline([
 
    (
        "scaler",
        StandardScaler()
    ),
 
    (
        "mlp",
        MLPRegressor(
            max_iter=1000,
            random_state=42
        )
    )
])

In [6]:
# =====================================================
# SEARCH SPACE
# =====================================================
 
space = [
 
    Integer(1, 2, name="n_layers"),
 
    Integer(50, 400, name="layer1"),
     
    Integer(25, 300, name="layer2"),
 
    Categorical(
        ["relu", "tanh"],
        name="activation"
    ),
 
    Real(
        1e-8,
        1e-2,
        prior="log-uniform",
        name="alpha"
    ),
 
    Real(
        1e-5,
        5e-2,
        prior="log-uniform",
        name="learning_rate_init"
    ),
 
    Integer(
        16,
        128,
        name="batch_size"
    )
]

In [7]:
# =====================================================
# GROUP KFOLD
# =====================================================
 
gkf = GroupKFold(
    n_splits=5
)

In [8]:
#Check folds
for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1):

    print(f"\n===== Fold {fold} =====")

    test_experiments = (
        df.iloc[test_idx]
        [["Source#", "Database#"]]
        .drop_duplicates()
        .sort_values(
            ["Source#", "Database#"]
        )
    )

    print("Test experiments:")
    print(test_experiments)


===== Fold 1 =====
Test experiments:
      Source#  Database#
1498        3          1
6103        5          3
6531        5         11
0           5         16

===== Fold 2 =====
Test experiments:
      Source#  Database#
52          1          1
3805        4          3
5991        5          1
6149        5          4
6195        5          5
6360        5          8
6640        5         13

===== Fold 3 =====
Test experiments:
      Source#  Database#
533         1          2
2713        4          1
6047        5          2
6303        5          7
6417        5          9
6588        5         12
6697        5         14

===== Fold 4 =====
Test experiments:
      Source#  Database#
1014        2          1
3324        4          2
4899        4          5

===== Fold 5 =====
Test experiments:
      Source#  Database#
1256        2          2
4418        4          4
5509        4          6
6252        5          6
6474        5         10
6754        5         15


In [9]:
#check for leaks
for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1):

    train_exp = set(
        zip(
            df.iloc[train_idx]["Source#"],
            df.iloc[train_idx]["Database#"]
        )
    )

    test_exp = set(
        zip(
            df.iloc[test_idx]["Source#"],
            df.iloc[test_idx]["Database#"]
        )
    )

    overlap = train_exp.intersection(test_exp)

    print(
        f"Fold {fold}: "
        f"{len(overlap)} overlapping experiments"
    )

Fold 1: 0 overlapping experiments
Fold 2: 0 overlapping experiments
Fold 3: 0 overlapping experiments
Fold 4: 0 overlapping experiments
Fold 5: 0 overlapping experiments


In [10]:
# =====================================================
# OPTIMIZER
# =====================================================
 
optimizer = Optimizer(
    dimensions=space,
    base_estimator="GP",
    acq_func="EI",
    random_state=42
)

In [ ]:
# =====================================================
# OPTIMIZATION SETTINGS
# =====================================================
 
N_ITER = 100
 
best_rmse = np.inf
best_params = None
 
history = []

# =====================================================
# OBJECTIVE FUNCTION
# =====================================================
 
for iteration in range(N_ITER):
 
    params = optimizer.ask()
 
    (
        n_layers,
        layer1,
        layer2,
        activation,
        alpha,
        learning_rate_init,
        batch_size
    ) = params
 
# ==========================================
# BUILD ARCHITECTURE
# ==========================================
 
    if n_layers == 1:
 
        hidden_layer_sizes = (
            layer1,
        )
 
    else:
 
        hidden_layer_sizes = (
            layer1,
            layer2
    )
 
    print("\n" + "=" * 80)
    print(f"ITERATION {iteration + 1}/{N_ITER}")
    print("=" * 80)
 
    print(
        f"Architecture = {hidden_layer_sizes}"
    )
 
    print(
        f"Activation = {activation}"
    )
 
    print(
        f"Alpha = {alpha:.8e}"
    )
 
    print(
        f"Learning Rate = {learning_rate_init:.8e}"
    )
 
    print(
        f"Batch Size = {batch_size}"
    )
 
    fold_rmse = []
    fold_mae = []
    fold_r2 = []
 
# ==========================================
# GROUP KFOLD CV
# ==========================================
 
    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups),
        start=1
    ):
 
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
 
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
 
# ======================================
# SCALE X
# ======================================
 
        x_scaler = StandardScaler()
 
        X_train_scaled = x_scaler.fit_transform(
            X_train
        )
 
        X_test_scaled = x_scaler.transform(
            X_test
    )
 
# ======================================
# SCALE y
# ======================================
 
        y_scaler = StandardScaler()
 
        y_train_scaled = (
            y_scaler.fit_transform(
                y_train.values.reshape(-1, 1)
            )
            .ravel()
        )
 
# ======================================
# MODEL
# ======================================
 
        model = MLPRegressor(
 
            hidden_layer_sizes=
                hidden_layer_sizes,
 
            activation=
                activation,
 
            alpha=
                alpha,
 
            learning_rate_init=
                learning_rate_init,
 
            batch_size=
                batch_size,
 
            solver="adam",
 
            max_iter=1000,
 
            early_stopping=True,
 
            validation_fraction=0.10,
 
            n_iter_no_change=30,
 
            random_state=42
        )
 
        model.fit(
            X_train_scaled,
            y_train_scaled
        )
 
# ======================================
# PREDICTIONS
# ======================================
 
        pred_scaled = model.predict(
            X_test_scaled
        )
 
        pred = (
            y_scaler.inverse_transform(
                pred_scaled.reshape(-1, 1)
            )
            .ravel()
        )
 
# ======================================
# METRICS
# ======================================
 
        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                pred
            )
        )
 
        mae = mean_absolute_error(
            y_test,
            pred
        )
 
        r2 = r2_score(
            y_test,
            pred
        )
 
        fold_rmse.append(rmse)
        fold_mae.append(mae)
        fold_r2.append(r2)
 
        print(
            f"Fold {fold}: "
            f"RMSE={rmse:.5f}, "
            f"MAE={mae:.5f}, "
            f"R2={r2:.5f}"
        )
 
# ==========================================
# MEAN PERFORMANCE
# ==========================================
 
    mean_rmse = np.mean(
        fold_rmse
    )
     
    mean_mae = np.mean(
        fold_mae
    )
 
    mean_r2 = np.mean(
        fold_r2
    )
 
# ==========================================
# TELL OPTIMIZER
# ==========================================
 
    optimizer.tell(
        params,
        mean_rmse
    )
 
# ==========================================
# SAVE HISTORY
# ==========================================
 
    history.append({
 
        "iteration":
            iteration + 1,
 
        "n_layers":
            n_layers,
 
        "layer1":
            layer1,
 
        "layer2":
            layer2,
 
        "hidden_layer_sizes":
            str(hidden_layer_sizes),
 
        "activation":
            activation,
 
        "alpha":
            alpha,
 
        "learning_rate_init":
            learning_rate_init,
 
        "batch_size":
            batch_size,
 
        "RMSE":
            mean_rmse,
 
        "MAE":
            mean_mae,
         
        "R2":
            mean_r2
    })
 
# ==========================================
# BEST RESULT
# ==========================================
 
    if mean_rmse < best_rmse:
 
        best_rmse = mean_rmse
 
        best_params = {
 
            "hidden_layer_sizes":
                hidden_layer_sizes,
 
            "activation":
                activation,
 
            "alpha":
                alpha,
 
            "learning_rate_init":
                learning_rate_init,
 
            "batch_size":
                batch_size
        }
 
    print(
        f"\nMean RMSE = {mean_rmse:.5f}"
    )
 
    print(
        f"Best RMSE = {best_rmse:.5f}"
    )


ITERATION 1/100
Architecture = (np.int64(114), np.int64(239))
Activation = tanh
Alpha = 4.73147464e-06
Learning Rate = 2.34317225e-05
Batch Size = 67


In [ ]:
# =====================================================
# FINAL RESULTS
# =====================================================
 
print("\n" + "=" * 80)
print("OPTIMIZATION COMPLETE")
print("=" * 80)
 
print(
    f"\nBest RMSE = {best_rmse:.5f}"
)
 
print(
    "\nBest Parameters:"
)
 
for k, v in best_params.items():
    print(
        f"{k}: {v}"
    )

In [ ]:
# =====================================================
# EVALUATE BEST HYPERPARAMETERS
# =====================================================
 
print("\n" + "=" * 80)
print("FINAL 5-FOLD EVALUATION")
print("=" * 80)
 
fold_results = []
 
all_true = []
all_pred = []
 
for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups),
    start=1
):
 
    print(f"\nEvaluating Fold {fold}")
 
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
     
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
 
# ==========================================
# SCALE FEATURES
# ==========================================
 
    x_scaler = StandardScaler()
 
    X_train_scaled = x_scaler.fit_transform(
        X_train
    )
 
    X_test_scaled = x_scaler.transform(
        X_test
    )
 
# ==========================================
# SCALE TARGET
# ==========================================
 
    y_scaler = StandardScaler()
 
    y_train_scaled = (
        y_scaler.fit_transform(
            y_train.values.reshape(-1, 1)
        )
    .ravel()
    )
 
# ==========================================
# TRAIN MODEL WITH BEST PARAMS
# ==========================================
 
    model = MLPRegressor(
 
        hidden_layer_sizes=
            best_params["hidden_layer_sizes"],
 
        activation=
            best_params["activation"],
 
        alpha=
            best_params["alpha"],
 
        learning_rate_init=
            best_params["learning_rate_init"],
 
        batch_size=
            best_params["batch_size"],
 
        max_iter=1000,
 
        early_stopping=True,
 
        validation_fraction=0.10,
 
        n_iter_no_change=30,
 
        random_state=42
    )
 
    model.fit(
        X_train_scaled,
        y_train_scaled
    )
 
# ==========================================
# PREDICT
# ==========================================
 
    pred_scaled = model.predict(
        X_test_scaled
    )
 
    pred = (
        y_scaler.inverse_transform(
            pred_scaled.reshape(-1, 1)
        )
    .ravel()
    )
 
# ==========================================
# METRICS
# ==========================================
 
    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )
 
    mae = mean_absolute_error(
        y_test,
        pred
    )
 
    r2 = r2_score(
        y_test,
        pred
    )
 
    fold_results.append({
        "Fold": fold,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })
 
    all_true.extend(
        y_test
    )
 
    all_pred.extend(
        pred
    )
 
    print(
        f"RMSE = {rmse:.5f}"
    )
 
    print(
        f"MAE = {mae:.5f}"
    )
 
    print(
        f"R² = {r2:.5f}"
    )

In [ ]:
# =====================================================
# OVERALL CV METRICS
# =====================================================
 
results_df = pd.DataFrame(
    fold_results
)
 
print("\n")
print(results_df)
 
print("\nMean Performance")
 
print(
    f"RMSE = {results_df['RMSE'].mean():.5f}"
    f" ± {results_df['RMSE'].std():.5f}"
)
 
print(
    f"MAE = {results_df['MAE'].mean():.5f}"
    f" ± {results_df['MAE'].std():.5f}"
)
 
print(
    f"R² = {results_df['R2'].mean():.5f}"
    f" ± {results_df['R2'].std():.5f}"
)

In [ ]:
# =====================================================
# FINAL TRAINING DATA
# =====================================================
 
x_scaler = StandardScaler()
y_scaler = StandardScaler()
 
X_scaled = x_scaler.fit_transform(X)
 
y_scaled = (
    y_scaler.fit_transform(
        y.values.reshape(-1, 1)
    )
    .ravel()
)

# =====================================================
# FINAL MODEL
# =====================================================
 
final_model = MLPRegressor(
    hidden_layer_sizes=
        best_params["hidden_layer_sizes"],
    activation=
        best_params["activation"],
    alpha=
        best_params["alpha"],
    learning_rate_init=
        best_params["learning_rate_init"],
    batch_size=
        best_params["batch_size"],
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=30,
    random_state=42
)
 
final_model.fit(
    X_scaled,
    y_scaled
)

In [ ]:
# =====================================================
# SAVE EVERYTHING
# =====================================================
 
joblib.dump(
    {
        "model": final_model,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "features": X.columns.tolist()
    },
    "MLP_dXdt_SkOptOptimizer.pkl"
)

In [ ]:
#Feature importance
from sklearn.inspection import permutation_importance
 
perm = permutation_importance(
    best_model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)
 
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
})
 
importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)
 
print("\nTop Features")
 
print(importance_df.head(20))

In [ ]:
# =====================================================
# BAR PLOT
# =====================================================
plt.figure(figsize=(10,8))
 
top_n = 20
 
plot_df = importance_df.head(top_n)
 
plt.barh(
    plot_df["Feature"],
    plot_df["Importance"]
)
 
plt.gca().invert_yaxis()
 
plt.xlabel("Permutation Importance")
plt.ylabel("Feature")
 
plt.tight_layout()
 
plt.show()

In [ ]:
# =====================================================
# SHAP
# =====================================================

import shap
 
print("\nCalculating SHAP values...")
 
explainer = shap.TreeExplainer(
    best_model
)
 
shap_values = explainer.shap_values(
    X
)

shap.summary_plot(
    shap_values,
    X
)